## Producer - poll CoinGecko, push ticks straight to Delta via Zerobus

Second transport next to `01_producer_price_ticks.ipynb` (Event Hub). Same CoinGecko request, same payload and the same bad-event injection, so the data-quality demo works for both transports. No message bus: records land in `ticks_zerobus_raw`, and the Lakeflow append flow `ticks_bronze_from_zerobus` moves them into `ticks_bronze`.

- Service principal credentials come from the Key Vault-backed secret scope (`team-crypto-zerobus-client-id`, `team-crypto-zerobus-client-secret`)
- Every record carries `producer_id` (one per run), `producer_seq` (1..n inside a run) and `producer_sent_at`, so lost or duplicated records can be found later
- `ingest_record_offset` only queues the record; after each poll the producer waits for the last offset, so everything sent so far is durable
- `NonRetriableException` (credentials, grants, schema) stops the producer with a clear message; the stream is always closed in `finally`

In [ ]:
%pip install databricks-zerobus-ingest-sdk
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "team_crypto_bronze")
dbutils.widgets.text("zerobus_table", "ticks_zerobus_raw")
# https://<workspace-id>.zerobus.<region>.azuredatabricks.net
dbutils.widgets.text("zerobus_endpoint", "")
dbutils.widgets.text("secret_scope", "team-crypto-scope")
dbutils.widgets.text("symbols", "bitcoin,ethereum,solana")
dbutils.widgets.text("poll_seconds", "10")
dbutils.widgets.text("iterations", "30")
# every N-th good event is followed by one bad event; 0 = off
dbutils.widgets.text("bad_every", "0")
# schema evolution twist: adds volume_24h to every event
dbutils.widgets.dropdown("include_volume", "false", ["false", "true"])

In [ ]:
import json
import logging
import time
from datetime import datetime, timezone

import requests
from zerobus.sdk.shared import (
    NonRetriableException,
    RecordType,
    StreamConfigurationOptions,
    TableProperties,
    ZerobusException,
)
from zerobus.sdk.sync import ZerobusSdk

logging.basicConfig(level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("zerobus_ticks_producer")
logger.setLevel(logging.INFO)

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
zerobus_table = dbutils.widgets.get("zerobus_table")
zerobus_endpoint = dbutils.widgets.get("zerobus_endpoint").strip()
secret_scope = dbutils.widgets.get("secret_scope")
symbols = dbutils.widgets.get("symbols").split(",")
poll_seconds = int(dbutils.widgets.get("poll_seconds"))
iterations = int(dbutils.widgets.get("iterations"))
bad_every = int(dbutils.widgets.get("bad_every"))
include_volume = dbutils.widgets.get("include_volume") == "true"

if not zerobus_endpoint:
    raise ValueError("Set the zerobus_endpoint widget")

workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
client_id = dbutils.secrets.get(scope=secret_scope, key="team-crypto-zerobus-client-id")
client_secret = dbutils.secrets.get(scope=secret_scope, key="team-crypto-zerobus-client-secret")
full_table = f"{catalog}.{bronze_schema}.{zerobus_table}"
producer_id = f"coingecko-ticks-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"

In [ ]:
sdk = ZerobusSdk(zerobus_endpoint, workspace_url)
stream = sdk.create_stream(
    client_id,
    client_secret,
    TableProperties(full_table),
    StreamConfigurationOptions(record_type=RecordType.JSON),
)

In [ ]:
# bad events for the data-quality demo: negative price, missing symbol, missing price, exact duplicate
BAD_EVENTS = [
    lambda e: {**e, "price_usd": -1.0},
    lambda e: {**e, "symbol": None},
    lambda e: {k: v for k, v in e.items() if k != "price_usd"},
    lambda e: dict(e),
]

producer_seq = 0


def send(event):
    """Queue one tick; returns the offset to wait for."""
    global producer_seq
    producer_seq += 1
    record = {
        "value": json.dumps(event),
        "producer_id": producer_id,
        "producer_seq": producer_seq,
        # TIMESTAMP goes over the JSON api as Unix microseconds
        "producer_sent_at": time.time_ns() // 1_000,
    }
    return stream.ingest_record_offset(json.dumps(record))


ids = ",".join(symbols)
good_sent = 0
bad_sent = 0
logger.info("Producer %s sends ticks for %s to %s", producer_id, ids, full_table)

try:
    for _ in range(iterations):
        resp = requests.get(
            "https://api.coingecko.com/api/v3/simple/price",
            params={"ids": ids, "vs_currencies": "usd", "include_24hr_vol": str(include_volume).lower()},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        now = time.time()
        offset = None
        for symbol in symbols:
            if symbol in data:
                event = {"symbol": symbol, "price_usd": data[symbol]["usd"], "ts": now}
                if include_volume:
                    event["volume_24h"] = data[symbol].get("usd_24h_vol")
                offset = send(event)
                good_sent += 1
                if bad_every and good_sent % bad_every == 0:
                    make_bad = BAD_EVENTS[(good_sent // bad_every - 1) % len(BAD_EVENTS)]
                    offset = send(make_bad(event))
                    bad_sent += 1
        # everything sent in this poll is durable before the next one
        if offset is not None:
            stream.wait_for_offset(offset)
        time.sleep(poll_seconds)
    stream.flush()
except NonRetriableException:
    logger.exception("Zerobus rejected the stream, check credentials, grants and the table schema")
    raise
except ZerobusException:
    logger.exception("Zerobus ingestion failed after the SDK retries")
    raise
finally:
    stream.close()

logger.info("Producer %s sent %d good and %d bad ticks (producer_seq 1..%d)", producer_id, good_sent, bad_sent, producer_seq)